# guppy and hugr to qir conversion and submission to H2


This example shows how to convert guppy to qir which can be submitted directly to H1 and H2 device, emulator and syntax checker.

You need to install hugr-qir, guppy and pytket-quantinuum for this notebook to work.

### Current guppy features that can't be converted:
- loops with condition not known at compiletime
- functions returning qubit arrays
- RNG functions
- dynamic qubit allocation


In [1]:
# You can write your guppy directly in a notebook or in a separate file
from typing import no_type_check

from guppylang import guppy, qubit
from guppylang.std.builtins import output
from guppylang.std.quantum import h, measure


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    h(q0)
    h(q1)

    b0 = measure(q0).read()
    b1 = measure(q1).read()
    b2 = b0 ^ b1

    output("0", b2)

# Convert hugr to qir

By default, the function will automatically check the generated QIR to capture most of the issues that could happen.
This will show an error message with more details about the problem the check can be turned off using the keyword argument `validate_qir = False`

In [2]:
from hugr_qir.guppy_to_qir import guppy_to_qir_str, guppy_to_qir_bytes
guppy_qir_bitcode_string, result_spec = guppy_to_qir_str(main)

In [3]:
# To get a human-readable LLVM assemly language string use the `hugr_to_qir` function with the keyword argument `output_format = OutputFormat.LLVM_IR`
from hugr_qir.guppy_to_qir import guppy_to_qir_str, guppy_to_qir_bytes

guppy_qir, result_spec = guppy_to_qir_str(main)
print(guppy_qir)

; ModuleID = 'hugr-qir'
source_filename = "hugr-qir"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i8:8:32-i16:16:32-i64:64-i128:128-n32:64-S128-Fn32"
target triple = "aarch64-unknown-linux-gnu"

@0 = private unnamed_addr constant [2 x i8] c"0\00", align 1
@gen_name = private unnamed_addr constant [8 x i8] c"hugr-qir", section ",qir_generator"
@gen_version = private unnamed_addr constant [10 x i8] c"0.2.0-rc.2", section ",qir_generator"

define void @__hugr__.__main__.main.1() local_unnamed_addr #0 {
alloca_block:
  tail call void @__quantum__rt__initialize(ptr null)
  tail call void @__quantum__qis__phasedx__body(double 0x3FF921FB54442D18, double 0xBFF921FB54442D18, ptr null)
  tail call void @__quantum__qis__rz__body(double 0x400921FB54442D18, ptr null)
  tail call void @__quantum__qis__mz__body(ptr null, ptr null)
  %0 = tail call i1 @__quantum__rt__read_result(ptr null)
  tail call void @__quantum__qis__phasedx__body(double 0x3FF921FB54442D18, double 0xBFF921FB54442D1

### Loops in the program are unrolled automatically when possible, because backwards branching is not available on H Series.  This means that the QIR generated from programs containing loops can get quite long as shown here:

In [4]:
from typing import no_type_check

from guppylang import guppy


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    for _ in range(10):
        q3 = qubit()
        h(q3)
        b = measure(q3).read()
        if b:
            h(q0)

    output("0", measure(q0).read())
    output("1", measure(q1).read())

In [5]:
guppy_qir, result_spec = guppy_to_qir_str(main, validate_qir=True)
print(guppy_qir)

; ModuleID = 'hugr-qir'
source_filename = "hugr-qir"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i8:8:32-i16:16:32-i64:64-i128:128-n32:64-S128-Fn32"
target triple = "aarch64-unknown-linux-gnu"

@0 = private unnamed_addr constant [2 x i8] c"0\00", align 1
@1 = private unnamed_addr constant [2 x i8] c"1\00", align 1
@gen_name = private unnamed_addr constant [8 x i8] c"hugr-qir", section ",qir_generator"
@gen_version = private unnamed_addr constant [10 x i8] c"0.2.0-rc.2", section ",qir_generator"

define void @__hugr__.__main__.main.1() local_unnamed_addr #0 {
alloca_block:
  tail call void @__quantum__rt__initialize(ptr null)
  tail call void @__quantum__qis__phasedx__body(double 0x3FF921FB54442D18, double 0xBFF921FB54442D18, ptr nonnull inttoptr (i64 2 to ptr))
  tail call void @__quantum__qis__rz__body(double 0x400921FB54442D18, ptr nonnull inttoptr (i64 2 to ptr))
  tail call void @__quantum__qis__mz__body(ptr nonnull inttoptr (i64 2 to ptr), ptr null)
  %0 = tail cal

### A similar example to the one above with a loop that terminates based on a measurement result inside the loop leads to the generation of QIR which is valid within the standard, but can't be executed on H-Series devices.

In [6]:
from typing import no_type_check

from guppylang import guppy


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    i = 0
    while i < 10:
        q3 = qubit()
        h(q3)
        b = measure(q3).read()
        if b:
            h(q0)
            i += 1

    output("0", measure(q0).read())
    output("1", measure(q1).read())

In [7]:
guppy_qir, result_spec = guppy_to_qir_str(main, validate_qir=False)
print(guppy_qir)

; ModuleID = 'hugr-qir'
source_filename = "hugr-qir"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i8:8:32-i16:16:32-i64:64-i128:128-n32:64-S128-Fn32"
target triple = "aarch64-unknown-linux-gnu"

@0 = private unnamed_addr constant [2 x i8] c"0\00", align 1
@1 = private unnamed_addr constant [2 x i8] c"1\00", align 1
@gen_name = private unnamed_addr constant [8 x i8] c"hugr-qir", section ",qir_generator"
@gen_version = private unnamed_addr constant [10 x i8] c"0.2.0-rc.2", section ",qir_generator"

define void @__hugr__.__main__.main.1() local_unnamed_addr #0 {
alloca_block:
  tail call void @__quantum__rt__initialize(ptr null)
  br label %cond_118_case_1

cond_exit_11:                                     ; preds = %cond_118_case_1, %bb0
  %"44_0.0" = phi i64 [ %4, %bb0 ], [ %"6_0.0145", %cond_118_case_1 ]
  %0 = icmp slt i64 %"44_0.0", 10
  br i1 %0, label %cond_118_case_1, label %bb

bb:                                               ; preds = %cond_exit_11
  tail call vo

In [8]:
# this will fail because of the loop in the generated QIR

try:
    guppy_qir, result_spec = guppy_to_qir_str(main)
    print(guppy_qir)
except Exception as e:
    print("Validation failed as expected:")
    print(e)

Validation failed as expected:
QIR generation failed. This may be the result of a bug but can also happen when trying to convert a feature in HUGR/Guppylang which is not supported in QIR. The failure occurred in the validity check of the generated QIR. This check can be disabled by setting `--no-validate-qir` on the cli or passing `validate_qir=False` for library calls. Error details: Found loop in CFG containing the block: cond_118_case_1


# Submission to the device via Nexus

The QIR generated can be submitted directly to Nexus. The python Nexus API is available via `pip install qnexus`. This requires a different QIR format for the submission, which can be generated from `compile_qir`.

In [9]:
import qnexus as qnx

qnx.login()

Already logged in. Tokens are valid.


In [10]:
import datetime

project = qnx.projects.get_or_create(name="QIR-Demonstration3")
qnx.context.set_active_project(project)

qir_name = "HUGR-QIR"
jobname_suffix = datetime.datetime.now().strftime("%Y_%m_%d-%H-%M-%S")

In [11]:
# You can write your guppy directly in a notebook or in a separate file
from typing import no_type_check

from guppylang import guppy


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    h(q0)
    h(q1)

    b0 = measure(q0).read()
    b1 = measure(q1).read()
    b2 = b0 ^ b1

    output("0", b2)

In [12]:
guppy_qir_bitcode, result_spec = guppy_to_qir_bytes(main)

In [13]:
qir_program_ref = qnx.qir.upload(qir=guppy_qir_bitcode, name=qir_name, project=project)

In [14]:
# Run on the H2-1 Syntax checker
device_name = "H2-1SC"

qnx.context.set_active_project(project)
config = qnx.QuantinuumConfig(device_name=device_name)

job_name = f"execution-job-qir-{qir_name}-{device_name}-{jobname_suffix}"
ref_execute_job = qnx.start_execute_job(
    programs=[qir_program_ref],
    n_shots=[10],
    backend_config=config,
    name=job_name,
)

In [15]:
qnx.jobs.wait_for(ref_execute_job)

JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 8, 20, 12, 39, 52, 294757, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 8, 20, 12, 39, 51, 748977, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 8, 20, 12, 39, 50, 555756, tzinfo=datetime.timezone.utc), running_time=None, cancelled_time=None, error_time=None, queue_position=None, cost=5.04)

In [16]:
qir_result = qnx.jobs.results(ref_execute_job)[0].download_result()

Unknown OpType in BackendInfo: `RZZ`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `Rxxyyzz`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `U1q`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `ZZ`, will omit from BackendInfo. Consider updating your pytket version.


In [17]:
qir_result.get_counts()

Counter({(0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0): 10})